# 🌐 Module 7.2: Local Model Serving

**Goal**: Serve an MLFlow model as a local REST API and send predictions via HTTP.

---

## Step 1: Train and Log a Model

First, let's create a model to serve.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import json
import requests

mlflow.autolog(disable=True)
mlflow.set_experiment("07_local_serving")

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="model_to_serve"):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(model, "model", signature=signature, input_example=X_train[:2])
    mlflow.log_metric("accuracy", accuracy_score(y_test, model.predict(X_test)))
    
    run_id = mlflow.active_run().info.run_id

print(f"✅ Model logged! Run ID: {run_id}")
print(f"\nTo serve this model, open a NEW terminal and run:")
print(f"  mlflow models serve -m runs:/{run_id}/model -p 1234 --no-conda")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## Step 2: Start the Model Server

⚠️ **Open a NEW terminal** and run the command printed above. Keep this notebook open.

```bash
cd c:\Users\sujat\projects\MLFlow_Learn
mlflow models serve -m runs:/<YOUR_RUN_ID>/model -p 1234 --no-conda
```

Wait for: `Listening at: http://127.0.0.1:1234`

## Step 3: Health Check

In [ ]:
# Check if the server is running
SERVER_URL = "http://127.0.0.1:1234"

try:
    response = requests.get(f"{SERVER_URL}/ping")
    print(f"✅ Server is running! Status: {response.status_code}")
except requests.ConnectionError:
    print("❌ Server is not running.")
    print("   Start it with the command in Step 2.")

## Step 4: Send Predictions

In [ ]:
# Prepare sample data
sample_data = X_test[:5]

# Format 1: dataframe_split (recommended)
payload = {
    "dataframe_split": {
        "columns": list(sample_data.columns),
        "data": sample_data.values.tolist()
    }
}

try:
    response = requests.post(
        f"{SERVER_URL}/invocations",
        headers={"Content-Type": "application/json"},
        json=payload
    )
    
    predictions = response.json()
    print("🔮 Predictions from REST API:")
    print(f"   Predicted: {predictions.get('predictions', predictions)}")
    print(f"   Actual:    {list(y_test[:5])}")
    print(f"\n✅ REST API is working!")
except requests.ConnectionError:
    print("❌ Could not connect. Make sure the server is running.")

In [ ]:
# Format 2: dataframe_records (alternative)
payload_records = {
    "dataframe_records": sample_data.to_dict(orient="records")
}

try:
    response = requests.post(
        f"{SERVER_URL}/invocations",
        headers={"Content-Type": "application/json"},
        json=payload_records
    )
    print(f"Records format response: {response.json()}")
except requests.ConnectionError:
    print("Server not running.")

## Step 5: Using curl (Command Line)

You can also test the API with curl from a terminal:

```bash
curl -X POST http://127.0.0.1:1234/invocations \
  -H "Content-Type: application/json" \
  -d '{"dataframe_split": {"columns": ["alcohol", "malic_acid", "ash", "alcalinity_of_ash", "magnesium", "total_phenols", "flavanoids", "nonflavanoid_phenols", "proanthocyanins", "color_intensity", "hue", "od280/od315_of_diluted_wines", "proline"], "data": [[13.0, 1.5, 2.3, 15.0, 100, 2.5, 3.0, 0.3, 1.5, 5.0, 1.0, 3.0, 1000]]}}'
```

## 🔑 Key Takeaways

1. **`mlflow models serve`** creates a REST API from any logged model
2. **`/invocations`** endpoint accepts JSON with `dataframe_split` or `dataframe_records` format
3. **`/ping`** for health checks, **`/version`** for server info
4. Use `--no-conda` to skip environment creation (uses current env)
5. The `requests` library makes it easy to test from Python

---

## ➡️ Next: `03_docker_containerization.ipynb` — Package in Docker